# Single-qubit gate error

*Exercises the GUIDE custom-`Mechanism` recipe: the core loop without the term layer.*

Target: ideal Rabi $\pi$-pulse, $H = \frac{\Omega}{2}\sigma_x$, driving $|0\rangle \to |1\rangle$ at $T = \pi/\Omega$.

Realized: same drive with amplitude/detuning error,
$H(t) = \frac{\Omega(1+\epsilon)}{2}\sigma_x + \delta\,\sigma_z$.

No subsystems here -- just `Operator` -> `Mechanism` -> `HamiltonianEvolution` -> fidelity.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from htdse.core.operator import Operator
from htdse.core.mechanism import Mechanism
from htdse.core.evolution import HamiltonianEvolution
from htdse.submodules.spin import sigma_x, sigma_z, I2
from htdse.util import ket, fidelity, process_fidelity

## Mechanism

$\epsilon$: fractional Rabi-rate (amplitude) error. $\delta$: detuning error. Ideal pulse is $\epsilon=\delta=0$.

Extends `Mechanism` (not just duck-typed) so shared behavior -- e.g. `__repr__` -- comes for free.

In [2]:
class RabiDrive(Mechanism):
    def __init__(self, Omega, eps=0.0, delta=0.0):
        self.Omega = Omega
        self.eps = eps
        self.delta = delta

    def hamiltonian(self, t) -> Operator:
        H = 0.5 * self.Omega * (1 + self.eps) * sigma_x + self.delta * sigma_z
        return Operator(H)

## Evolve target vs. realized

Propagators come from evolving $I$; states come from evolving $|0\rangle$. `HamiltonianEvolution`
only takes a start time `t0` (default 0) -- it solves exactly the range each `state_at` call
asks for, printing every real integration it performs (`verbose=True` by default), so nothing
is ever silently interpolated/extrapolated past solved data.

**If you want every evolution, not just one initial state's:** `initial` has no shape
restriction, so evolving $I$ once already gives you $U(t)$ at every requested time --
apply it to as many initial states as you like afterward via `U(t) @ psi0`, no re-solving.
Equivalently, you can evolve several kets at once in a single solve by stacking them as
columns of `initial` and passing that whole matrix in, since $H(t)X$ acts on every column
simultaneously.

In [3]:
Omega = 1.0
eps, delta = 0.05, 0.02
T = np.pi / Omega  # ideal pi-pulse time

target = RabiDrive(Omega)
realized = RabiDrive(Omega, eps=eps, delta=delta)

U_target = HamiltonianEvolution(target, Operator(I2)).state_at(T)
U_real = HamiltonianEvolution(realized, Operator(I2)).state_at(T)

psi_target = HamiltonianEvolution(target, Operator(ket("0"))).state_at(T)
psi_real = HamiltonianEvolution(realized, Operator(ket("0"))).state_at(T)

print(f"process fidelity: {process_fidelity(U_target, U_real):.6f}")
print(f"state fidelity:   {fidelity(psi_target, psi_real):.6f}")

[HamiltonianEvolution] integrating RabiDrive(Omega=1.0, eps=0.0, delta=0.0): t=0 -> 3.14159, method=RK45, rtol=1e-08, atol=1e-10
[HamiltonianEvolution]   done: success=True, steps=22, rhs evals=128
[HamiltonianEvolution] integrating RabiDrive(Omega=1.0, eps=0.05, delta=0.02): t=0 -> 3.14159, method=RK45, rtol=1e-08, atol=1e-10
[HamiltonianEvolution]   done: success=True, steps=24, rhs evals=146
[HamiltonianEvolution] integrating RabiDrive(Omega=1.0, eps=0.0, delta=0.0): t=0 -> 3.14159, method=RK45, rtol=1e-08, atol=1e-10
[HamiltonianEvolution]   done: success=True, steps=22, rhs evals=128
[HamiltonianEvolution] integrating RabiDrive(Omega=1.0, eps=0.05, delta=0.02): t=0 -> 3.14159, method=RK45, rtol=1e-08, atol=1e-10
[HamiltonianEvolution]   done: success=True, steps=24, rhs evals=146
process fidelity: 0.992216
state fidelity:   0.992216


## Population trajectory

$|\langle 1|\psi(t)\rangle|^2$ under the realized drive, over an array of times. Each new
`evolution` instance starts unsolved; the prints below show it actually integrating 0 -> T
once, on demand, rather than assuming any pre-solved range.

In [ ]:
ts = np.linspace(0, T, 50)
evolution = HamiltonianEvolution(realized, Operator(ket("0")))
psis = evolution.state_at(ts)
pop1 = np.abs(psis[:, 1]) ** 2

plt.plot(ts, pop1)
plt.xlabel("t")
plt.ylabel(r"$|\langle 1|\psi(t)\rangle|^2$")
plt.title("Realized population transfer")
plt.show()

## Requesting an unsolved time just extends the solve

No guessing: asking for $t$ beyond what's been solved so far triggers a real continuation
integration from the last solved boundary state, printed below, not extrapolation.

In [ ]:
print("already solved up to T =", T)
psi_far = evolution.state_at(5 * T)
print("population at 5T:", np.abs(psi_far[1]) ** 2)